In [ ]:
import numpy as np
import pandas as pd


# -----------------------------
# Scenario settings
# -----------------------------
Ssizes = [300]
H2_Refill_Price_Range = [7.5]

Initial_storage_status = 0

simulation_years = 14
Lifetime = 30
Discount_Rate = 0.05


# -----------------------------
# Load demand data
# -----------------------------
PV = pd.read_csv("PVhourlyHydrogen500.csv", delimiter=";")


# -----------------------------
# Helper values
# -----------------------------
discount_factor = sum(
    1 / (1 + Discount_Rate) ** t
    for t in range(1, Lifetime + 1)
)

economic_df = pd.DataFrame()


# -----------------------------
# Main loop
# -----------------------------
for H2_Refill_Price in H2_Refill_Price_Range:
    for Ssize in Ssizes:

        refill_trigger = 0.1 * Ssize
        refill_target = Ssize

        storage_status = [Initial_storage_status]
        refill_happened = []
        refill_amounts = []
        refill_times = []
        h2_deficit_list = []

        Simh = PV[
            ["Year", "Month", "Day", "Hour", "Production (Wh)", "Consumption (kWh H2"]
        ].copy()

        Simh["Consumption (kg H2)"] = Simh["Consumption (kWh H2"] / 33.3

        for i in range(len(Simh)):
            prev_storage = storage_status[-1]
            demand_h2 = Simh.loc[i, "Consumption (kg H2)"]

            # Hydrogen available before demand
            available_h2 = prev_storage

            # Deficit before refill
            deficit = max(0, demand_h2 - available_h2)
            h2_deficit_list.append(deficit)

            # Storage after demand
            new_storage = max(0, available_h2 - demand_h2)

            refill_flag = 0
            refill_amount = 0

            if new_storage <= refill_trigger:
                refill_flag = 1
                refill_amount = refill_target - new_storage
                storage_before_refill = new_storage
                new_storage = refill_target

                refill_times.append({
                    "Year": Simh.loc[i, "Year"],
                    "Month": Simh.loc[i, "Month"],
                    "Day": Simh.loc[i, "Day"],
                    "Hour": Simh.loc[i, "Hour"],
                    "Storage before refill (kg H2)": storage_before_refill,
                    "Refill amount (kg H2)": refill_amount,
                    "Storage after refill (kg H2)": new_storage,
                })

            storage_status.append(new_storage)
            refill_happened.append(refill_flag)
            refill_amounts.append(refill_amount)

        # -----------------------------
        # Add simulation results
        # -----------------------------
        Simh["H2 storage (kg H2)"] = storage_status[1:]
        Simh["Refill happened"] = refill_happened
        Simh["Refill amount (kg H2)"] = refill_amounts
        Simh["H2 Refill Cost (EUR)"] = Simh["Refill amount (kg H2)"] * H2_Refill_Price
        Simh["H2def (kg H2)"] = h2_deficit_list

        # Zero values because this scenario has no PV, battery, electrolyzer, or grid production
        Simh["PV production (kWh)"] = 0
        Simh["Excess e (kWh)"] = 0
        Simh["Electrolyzer production (kWh)"] = 0
        Simh["PV to Electrolyzer (kWh)"] = 0
        Simh["Grid to Electrolyzer (kWh)"] = 0
        Simh["Hydrogen produced (kg)"] = 0
        Simh["Compressor flow (kg H2)"] = 0
        Simh["Battery status (kWh)"] = 0

        Simh.to_csv("Simh.csv", index=False, sep=";")

        refill_events_df = pd.DataFrame(refill_times)
        refill_events_df.to_csv("refill_events.csv", index=False, sep=";")

        # -----------------------------
        # Economic calculations
        # -----------------------------
        H2_Storage_Capex = 1100 * Ssize
        Annual_H2_Storage_Opex = 0.01 * H2_Storage_Capex

        Compressor_Capex = 140000
        Annual_Compressor_Opex = 0.005 * Compressor_Capex

        Battery_Capex = 0
        Annual_Battery_Opex = 0

        PV_Capex = 0
        Annual_PV_Opex = 0

        Electrolyzer_CAPEX = 0
        Annual_Electrolyzer_Opex = 0
        Total_Discounted_Replacement_Cost = 0

        Annual_N4_Electricity_Fixed_Cost = 0
        Total_Discounted_Electricity_Cost = 0
        Total_Discounted_Electricity_Income = 0
        Total_Discounted_Water_Cost = 0

        Total_CAPEX = (
            H2_Storage_Capex
            + Compressor_Capex
            + Battery_Capex
            + PV_Capex
            + Electrolyzer_CAPEX
        )

        Total_annual_OPEX_fix = (
            Annual_H2_Storage_Opex
            + Annual_Compressor_Opex
            + Annual_Battery_Opex
            + Annual_PV_Opex
            + Annual_Electrolyzer_Opex
            + Annual_N4_Electricity_Fixed_Cost
        )

        Total_Discounted_OPEX_fix = Total_annual_OPEX_fix * discount_factor

        Annual_H2_Refill_Cost = Simh["H2 Refill Cost (EUR)"].sum() / simulation_years
        Total_Discounted_H2_Refill_Cost = Annual_H2_Refill_Cost * discount_factor

        Annual_H2_Produced = 0
        Total_H2_Produced = 0

        Net_Present_Cost = (
            Total_CAPEX
            + Total_Discounted_OPEX_fix
            + Total_Discounted_Replacement_Cost
            + Total_Discounted_Electricity_Cost
            + Total_Discounted_Water_Cost
            + Total_Discounted_H2_Refill_Cost
            - Total_Discounted_Electricity_Income
        )

        LCOH = np.nan

        total_refill_h2 = Lifetime * Simh["Refill amount (kg H2)"].sum() / simulation_years
        annual_refill_h2 = Simh["Refill amount (kg H2)"].sum() / simulation_years

        economic_results = {
            "Storage Size (kg H2)": Ssize,
            "Total H2 refill purchased (kg)": total_refill_h2,
            "Annual H2 refill purchased (kg/year)": annual_refill_h2,
            "H2 refill price (EUR/kg H2)": H2_Refill_Price,
            "Total Discounted H2 Refill Cost (EUR2024)": Total_Discounted_H2_Refill_Cost,
            "Discount Rate": Discount_Rate,
            "Net Present Cost (EUR2024)": Net_Present_Cost,
            "Total CAPEX (EUR2024)": Total_CAPEX,
            "H2 Storage Capex": H2_Storage_Capex,
            "Compressor Capex": Compressor_Capex,
            "Total Discounted Fixed OPEX (EUR2024)": Total_Discounted_OPEX_fix,
            "Annual Electrolyzer Opex": Annual_Electrolyzer_Opex,
            "Annual PV Opex": Annual_PV_Opex,
            "Annual Compressor Opex": Annual_Compressor_Opex,
            "Annual Battery Opex": Annual_Battery_Opex,
            "Annual H2 Storage Opex": Annual_H2_Storage_Opex,
            "Annual N4 Electricity Fixed Cost": Annual_N4_Electricity_Fixed_Cost,
            "Total Discounted Replacement Cost (EUR2024)": Total_Discounted_Replacement_Cost,
            "Total Discounted Electricity Cost (EUR2024)": Total_Discounted_Electricity_Cost,
            "Total Discounted Water Cost (EUR2024)": Total_Discounted_Water_Cost,
            "Total Discounted Electricity Income (EUR2024)": Total_Discounted_Electricity_Income,
            "Total H2 produced (kg)": Total_H2_Produced,
            "Total H2 deficit (kg)": Lifetime * Simh["H2def (kg H2)"].sum() / simulation_years,
            "Electrolyzer usage (%)": 0,
            "Electrolyzer production (kWh)": 0,
            "PV to Electrolyzer (kWh)": 0,
            "Grid to Electrolyzer (kWh)": 0,
            "Consumption (kg)": Lifetime * Simh["Consumption (kg H2)"].sum() / simulation_years,
            "PV production (kWh)": 0,
            "Excess electricity": 0,
            "Number of refills": Lifetime * Simh["Refill happened"].sum() / simulation_years,
            "Total purchased refill H2 (kg)": total_refill_h2,
        }

        economic_df = pd.concat(
            [economic_df, pd.DataFrame([economic_results])],
            ignore_index=True
        )


economic_df.to_csv("economic_results.csv", index=False, sep=";")